# Ascent Sampling

This tutorial demonstrates how to use Ascent's `sample` pipeline to
resample a mesh onto a new grid or topology.

Sampling is useful when you need to:

- Reduce the resolution of a dataset for visualization.
- Convert an unstructured mesh to a uniform grid.
- Restrict sampling to a spatial region.
- Control the output resolution using dimensions or spacing.
- Sample data onto a custom topology.

The examples use Ascent's built-in `braid` example mesh.

In [ ]:
# conduit + ascent imports
import conduit
import ascent

# cleanup any old results
!./cleanup.sh

### Create an example mesh to feed to Ascent

In [ ]:
# create example mesh using the conduit blueprint braid helper
mesh = conduit.Node()
conduit.blueprint.mesh.examples.braid("hexs",
                                      20,
                                      20,
                                      20,
                                      mesh)

## Baseline: Render the Original Mesh

Before applying a sampling filter, render the original mesh. This provides
a baseline for comparing the sampled results.

The pseudocolor plot displays the scalar field named `braid`.

In [ ]:
# Use Ascent to bin an input mesh in a few ways
a = ascent.Ascent()

# open ascent
a.open()

# publish mesh to ascent
a.publish(mesh)

# setup actions
actions = conduit.Node()
add_act = actions.append()
add_act["action"] = "add_scenes"

scenes = add_act["scenes"]
scenes["s1/plots/p1/type"] = "pseudocolor"
scenes["s1/plots/p1/field"] = "braid"
scenes["s1/image_name"] = "un_sampled_data"

# view our full actions tree
print(actions.to_yaml())

# execute the actions
a.execute(actions)

In [ ]:
# show the resulting image
ascent.jupyter.AscentViewer(a).show()

In [ ]:
# close ascent
a.close()

## Sampling Example 1: Uniform Grid Dimensions

This example samples the `braid` field onto a uniform grid with dimensions
of 10 x 10 x 10.

The sampling region is explicitly defined by:

- Origin: `(-10, -10, -10)`
- Grid dimensions: `10 x 10 x 10`
- Grid spacing: `(1, 1, 1)`

Values that cannot be sampled are assigned `-10.0`.

In [ ]:
# Use Ascent to bin an input mesh in a few ways
a = ascent.Ascent()

# open ascent
a.open()

# publish mesh to ascent
a.publish(mesh)

# setup actions
actions = conduit.Node()

# Add a sampling pipeline
add_sample_act = actions.append()
add_sample_act["action"] = "add_pipelines"

sample_pipe = add_sample_act["pipelines"]
sample_pipe["pl1/f1/type"] = "sample"
sample_pipe["pl1/f1/params/fields"] = ["braid"]

# Define the output uniform grid
sample_pipe["pl1/f1/params/uniform_grid/dims/i"] = 10
sample_pipe["pl1/f1/params/uniform_grid/dims/j"] = 10
sample_pipe["pl1/f1/params/uniform_grid/dims/k"] = 10

sample_pipe["pl1/f1/params/uniform_grid/origin/x"] = -10
sample_pipe["pl1/f1/params/uniform_grid/origin/y"] = -10
sample_pipe["pl1/f1/params/uniform_grid/origin/z"] = -10

sample_pipe["pl1/f1/params/uniform_grid/spacing/dx"] = 1
sample_pipe["pl1/f1/params/uniform_grid/spacing/dy"] = 1
sample_pipe["pl1/f1/params/uniform_grid/spacing/dz"] = 1

sample_pipe["pl1/f1/params/invalid_value"] = -10.0

# Add a scene that renders the sampled result.
add_act = actions.append()
add_act["action"] = "add_scenes"

scenes = add_act["scenes"]
scenes["s1/plots/p1/type"] = "pseudocolor"
scenes["s1/plots/p1/field"] = "braid"
scenes["s1/plots/p1/pipeline"] = "pl1"
scenes["s1/image_name"] = "sample_uniform_grid"

# view our full actions tree
print(actions.to_yaml())

# execute the actions
a.execute(actions)

In [ ]:
# show the resulting image
ascent.jupyter.AscentViewer(a).show()

In [ ]:
# close ascent
a.close()

## Sampling Example 2: Uniform Grid Spacing

This example specifies the output resolution using grid spacing rather than
the number of grid points.

A spacing of `0.5` is used in each direction. Smaller spacing generally
produces a higher-resolution sampled grid and may require more memory and
processing time.

In [ ]:
# Use Ascent to bin an input mesh in a few ways
a = ascent.Ascent()

# open ascent
a.open()

# publish mesh to ascent
a.publish(mesh)

# setup actions
actions = conduit.Node()

# Add a sampling pipeline
add_sample_act = actions.append()
add_sample_act["action"] = "add_pipelines"

sample_pipe = add_sample_act["pipelines"]
sample_pipe["pl1/f1/type"] = "sample"
sample_pipe["pl1/f1/params/fields"] = ["braid"]

# Define the output uniform grid spacing
sample_pipe["pl1/f1/params/uniform_grid/spacing/dx"] = 0.5
sample_pipe["pl1/f1/params/uniform_grid/spacing/dy"] = 0.5
sample_pipe["pl1/f1/params/uniform_grid/spacing/dz"] = 0.5
sample_pipe["pl1/f1/params/invalid_value"] = -10.0

# Add a scene that renders the sampled result.
add_act = actions.append()
add_act["action"] = "add_scenes"

scenes = add_act["scenes"]
scenes["s1/plots/p1/type"] = "pseudocolor"
scenes["s1/plots/p1/field"] = "braid"
scenes["s1/plots/p1/pipeline"] = "pl1"
scenes["s1/image_name"] = "sample_uniform_grid_spacing"

# view our full actions tree
print(actions.to_yaml())

# execute the actions
a.execute(actions)

In [ ]:
# show the resulting image
ascent.jupyter.AscentViewer(a).show()

In [ ]:
# close ascent
a.close()

## Sampling Example 3: Sampling Within a Bounding Box

This example defines the sampling region using a bounding box rather than
an origin and spacing.

The output region is:

- Minimum corner: `(0, 0, 0)`
- Maximum corner: the maximum coordinate of the input mesh
- Resolution: `25 x 25 x 25`

This approach is useful when the sampling region should follow the spatial
extent of the input data.

In [ ]:
# Use Ascent to bin an input mesh in a few ways
a = ascent.Ascent()

# open ascent
a.open()

# publish mesh to ascent
a.publish(mesh)

# setup actions
actions = conduit.Node()

# Add a sampling pipeline
add_sample_act = actions.append()
add_sample_act["action"] = "add_pipelines"

sample_pipe = add_sample_act["pipelines"]
sample_pipe["pl1/f1/type"] = "sample"
sample_pipe["pl1/f1/params/fields"] = ["braid"]

# Define the bounding box
sample_pipe["pl1/f1/params/box/dims/i"] = 25.0
sample_pipe["pl1/f1/params/box/dims/j"] = 25.0
sample_pipe["pl1/f1/params/box/dims/k"] = 25.0

sample_pipe["pl1/f1/params/box/min/x"] = 0.0
sample_pipe["pl1/f1/params/box/min/y"] = 0.0
sample_pipe["pl1/f1/params/box/min/z"] = 0.0

sample_pipe["pl1/f1/params/box/max/x"] = "max"
sample_pipe["pl1/f1/params/box/max/y"] = "max"
sample_pipe["pl1/f1/params/box/max/z"] = "max"

sample_pipe["pl1/f1/params/invalid_value"] = -10.0

# Add a scene that renders the sampled result.
add_act = actions.append()
add_act["action"] = "add_scenes"

scenes = add_act["scenes"]
scenes["s1/plots/p1/type"] = "pseudocolor"
scenes["s1/plots/p1/field"] = "braid"
scenes["s1/plots/p1/pipeline"] = "pl1"
scenes["s1/image_name"] = "sample_bounding_box"

# view our full actions tree
print(actions.to_yaml())

# execute the actions
a.execute(actions)

In [ ]:
# show the resulting image
ascent.jupyter.AscentViewer(a).show()

In [ ]:
# close ascent
a.close()

## Sampling Example 4: Sampling Onto a Custom Topology

This example creates a spherical triangular topology and samples the `braid`
field onto it.

The sphere consists of:

- One north-pole vertex.
- Multiple rings of latitude vertices.
- One south-pole vertex.
- Triangles connecting the poles and adjacent latitude rings.

The longitude index wraps from the final longitude back to longitude zero.
This prevents a gap or twisted strip at the seam.

In [ ]:
# Add a new Spherical topology to the mesh


import math

radius = 10.0
num_lat = 12
num_lon = 24

############################################
### Defining the Topology Coordinate Set ###
############################################

# Initialize with north pole values
x_vals, y_vals, z_vals = [0.0], [0.0], [radius]

# Add lat, lon coordinates
for lat in range(1, num_lat):
    theta = math.pi * float(lat) / float(num_lat)
    sin_theta = math.sin(theta)
    cos_theta = math.cos(theta)

    for lon in range(num_lon):
        phi = 2.0 * math.pi * float(lon) / float(num_lon)

        x_vals.append(radius * sin_theta * math.cos(phi))
        y_vals.append(radius * sin_theta * math.sin(phi))
        z_vals.append(radius * cos_theta)

# Add south pole values
x_vals.append(0.0)
y_vals.append(0.0)
z_vals.append(-radius)

# Add new sphere coordset to mesh
coords = mesh["coordsets/sample_sphere_coords"]
coords["type"] = "explicit"
coords["values/x"] = x_vals
coords["values/y"] = y_vals
coords["values/z"] = z_vals


#########################################
### Defining the Topology Conectivity ###
#########################################

ring_idx = (lambda lat, lon: int(1 + (lat - 1) * num_lon + (lon % num_lon)))

conn = []
for lon in range(num_lon):
    conn.extend([0, ring_idx(1, lon + 1), ring_idx(1, lon)])

for lat in range(1, num_lat - 1):
    for lon in range(num_lon):
        lower_left = ring_idx(lat, lon)
        lower_right = ring_idx(lat, lon + 1)
        upper_left = ring_idx(lat + 1, lon)
        upper_right = ring_idx(lat + 1, lon + 1)

        conn.extend([lower_left, lower_right, upper_left,
                     lower_right, upper_right, upper_left])

for lon in range(num_lon):
    conn.extend([ring_idx(num_lat - 1, lon), ring_idx(num_lat - 1, lon + 1), int(len(x_vals))])

# Add sphere topology connectivity to mesh
topo = mesh["topologies/sample_sphere"]
topo["type"] = "unstructured"
topo["coordset"] = "sample_sphere_coords"
topo["elements/shape"] = "tri"
topo["elements/connectivity"] = conn

print(mesh.to_yaml())

In [ ]:
# Use Ascent to bin an input mesh in a few ways
a = ascent.Ascent()

# open ascent
a.open()

# publish mesh to ascent
a.publish(mesh)

# setup actions
actions = conduit.Node()

# Add a sampling pipeline
add_sample_act = actions.append()
add_sample_act["action"] = "add_pipelines"

sample_pipe = add_sample_act["pipelines"]
sample_pipe["pl1/f1/type"] = "sample"
sample_pipe["pl1/f1/params/fields"] = ["braid"]

# Define the topology to sample onto
sample_pipe["pl1/f1/params/topology"] = "sample_sphere"
sample_pipe["pl1/f1/params/invalid_value"] = -10.0

# Add a scene that renders the sampled result
add_act = actions.append()
add_act["action"] = "add_scenes"

scenes = add_act["scenes"]
scenes["s1/plots/p1/type"] = "pseudocolor"
scenes["s1/plots/p1/field"] = "braid"
scenes["s1/plots/p1/pipeline"] = "pl1"
scenes["s1/image_name"] = "sample_spherical_topology"

# view our full actions tree
print(actions.to_yaml())

# execute the actions
a.execute(actions)

In [ ]:
# show the resulting image
ascent.jupyter.AscentViewer(a).show()

In [ ]:
# close ascent
a.close()